In [6]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install rank-bm25
!pip install pandas
!pip install numpy
!pip install ibm-watsonx-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 131.2 MB/s  0:00:00


In [7]:
# ============================
# CELL 1: IMPORTS & SETUP
# ============================
# This cell loads all required libraries for the Enterprise Knowledge Navigator system.
# It includes document processing, embeddings, vector search, and LLM integration tools.

import io
import re
import faiss
import numpy as np
import ibm_boto3

from botocore.client import Config
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from ibm_watsonx_ai.foundation_models import ModelInference

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [8]:
# ============================
# CELL 2: IBM CLOUD COS CONNECTION
# ============================
# This cell establishes a secure connection to IBM Cloud Object Storage (COS)
# where all enterprise documents (PDFs) are stored.

cos_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id='',  # replace in production
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud'
)

bucket = ""

print("✅ COS connection established")

✅ COS connection established


In [9]:
# ============================
# CELL 3: LOAD PDF FILE LIST
# ============================
# This cell defines all medical / enterprise PDFs that will be processed
# by the Knowledge Navigator ingestion pipeline.

pdf_files = [
    "unilever-annual-report-and-accounts-2025.pdf",
    "Pexip_Infinity_Release_Notes_v40.a.pdf",
    "HR-Management-Handbook-for-SAIs-2019.pdf",
    "04-Policy-Manual-Apr-2016-excl-SAICA.pdf",
    "wcms_896633.pdf",
    "Human_Resources_Manual.pdf"
]

print(f"📄 Total documents loaded for ingestion: {len(pdf_files)}")

📄 Total documents loaded for ingestion: 6


In [10]:
# ============================
# CELL 4: PDF LOADER
# ============================
# This function downloads a PDF from IBM COS and returns raw bytes.

def load_pdf(bucket, key):
    response = cos_client.get_object(Bucket=bucket, Key=key)
    return response["Body"].read()

print("✅ PDF loader ready")

✅ PDF loader ready


In [11]:
# ============================
# CELL 5: PDF TEXT EXTRACTION
# ============================
# This function extracts clean text from PDF bytes using PyPDF.

def extract_text(pdf_bytes):
    pdf_file = io.BytesIO(pdf_bytes)
    reader = PdfReader(pdf_file)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text

print("✅ PDF text extractor ready")

✅ PDF text extractor ready


In [13]:
# ============================
# CELL 6: DOCUMENT INGESTION
# ============================
# This cell loads all PDFs from COS and extracts raw text into memory.
# Output is a structured list of documents.

documents = []

for file in pdf_files:
    print(f"📥 Processing: {file}")

    pdf_bytes = load_pdf(bucket, file)
    text = extract_text(pdf_bytes)

    documents.append({
        "source": file,
        "text": text
    })

print(f"✅ Total documents processed: {len(documents)}")

📥 Processing: unilever-annual-report-and-accounts-2025.pdf
📥 Processing: Pexip_Infinity_Release_Notes_v40.a.pdf
📥 Processing: HR-Management-Handbook-for-SAIs-2019.pdf
📥 Processing: 04-Policy-Manual-Apr-2016-excl-SAICA.pdf
📥 Processing: wcms_896633.pdf
📥 Processing: Human_Resources_Manual.pdf
✅ Total documents processed: 6


In [14]:
# ============================
# CELL 7: DOCUMENT CHUNKING
# ============================
# This cell splits long medical documents into smaller overlapping chunks
# to improve retrieval accuracy in FAISS + BM25.

def chunk_text(text, chunk_size=800):
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Apply chunking to all documents
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "metadata": {
                "source": doc["source"],
                "chunk_id": i
            }
        })

print(f"📦 Total chunks created: {len(all_chunks)}")

📦 Total chunks created: 2745


In [15]:
# ============================
# CELL 8: GENERATE EMBEDDINGS
# ============================
# This cell converts all chunks into vector embeddings
# using the SentenceTransformer model.
#
# Output:
# - chunk_embeddings
# - shape = (num_chunks, embedding_dimension)

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Extract text only
texts = [
    chunk["text"]
    for chunk in all_chunks
]

print("Texts:", len(texts))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Texts: 2745


In [16]:
# ============================
# CELL 9: EMBEDDING GENERATION
# ============================
# Generate embeddings in batches
# to avoid notebook memory issues.

batch_size = 32

chunk_embeddings = []

for i in range(0, len(texts), batch_size):

    batch = texts[i:i + batch_size]

    print(
        f"Embedding batch {i} -> {i + len(batch)}"
    )

    embeddings = embedding_model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    chunk_embeddings.append(
        embeddings
    )

chunk_embeddings = np.vstack(
    chunk_embeddings
).astype("float32")

print(
    "Embeddings shape:",
    chunk_embeddings.shape
)

Embedding batch 0 -> 32
Embedding batch 32 -> 64
Embedding batch 64 -> 96
Embedding batch 96 -> 128
Embedding batch 128 -> 160
Embedding batch 160 -> 192
Embedding batch 192 -> 224
Embedding batch 224 -> 256
Embedding batch 256 -> 288
Embedding batch 288 -> 320
Embedding batch 320 -> 352
Embedding batch 352 -> 384
Embedding batch 384 -> 416
Embedding batch 416 -> 448
Embedding batch 448 -> 480
Embedding batch 480 -> 512
Embedding batch 512 -> 544
Embedding batch 544 -> 576
Embedding batch 576 -> 608
Embedding batch 608 -> 640
Embedding batch 640 -> 672
Embedding batch 672 -> 704
Embedding batch 704 -> 736
Embedding batch 736 -> 768
Embedding batch 768 -> 800
Embedding batch 800 -> 832
Embedding batch 832 -> 864
Embedding batch 864 -> 896
Embedding batch 896 -> 928
Embedding batch 928 -> 960
Embedding batch 960 -> 992
Embedding batch 992 -> 1024
Embedding batch 1024 -> 1056
Embedding batch 1056 -> 1088
Embedding batch 1088 -> 1120
Embedding batch 1120 -> 1152
Embedding batch 1152 -> 118

In [17]:
# ============================
# CELL 10: BUILD FAISS INDEX
# ============================
#
# Creates a vector database
# for semantic similarity search.
#
# Input:
#   chunk_embeddings
#
# Output:
#   index
#
# ============================

import faiss


dimension = chunk_embeddings.shape[1]


index = faiss.IndexFlatL2(
    dimension
)


index.add(
    chunk_embeddings
)


print("FAISS ready")

print(
    "Vectors stored:",
    index.ntotal
)

print(
    "Expected:",
    len(all_chunks)
)

FAISS ready
Vectors stored: 2745
Expected: 2745


In [18]:
# ============================
# CELL 11: BM25 INDEX
# ============================
#
# Builds keyword search
# over document chunks.
#
# ============================


from rank_bm25 import BM25Okapi


tokenized_chunks = [

    chunk["text"].lower().split()

    for chunk in all_chunks

]


bm25 = BM25Okapi(

    tokenized_chunks

)


print("BM25 ready")

print(

    "Indexed:",

    len(all_chunks)

)

BM25 ready
Indexed: 2745


In [19]:
# ============================
# CELL 12: HYBRID SEARCH
# ============================
#
# Combines semantic search
# with keyword search.
#
# ============================


def hybrid_search(

        query,

        k=5

):


    # -------------------
    # FAISS
    # -------------------

    qvec = embedding_model.encode(

        [query],

        convert_to_numpy=True

    ).astype("float32")



    distances,indices = index.search(

        qvec,

        k

    )


    faiss_docs = [

        all_chunks[i]

        for i in indices[0]

    ]


    # -------------------
    # BM25
    # -------------------

    scores = bm25.get_scores(

        query.lower().split()

    )



    top_idx = sorted(

        range(

            len(scores)

        ),

        key=lambda i:scores[i],

        reverse=True

    )[:k]



    bm25_docs = [

        all_chunks[i]

        for i in top_idx

    ]



    # -------------------
    # MERGE
    # -------------------

    merged = {}


    for doc in faiss_docs + bm25_docs:


        key = (

            doc["metadata"]["source"],

            doc["metadata"]["chunk_id"]

        )


        merged[key] = doc



    return list(

        merged.values()

    )[:k]



print(

"Hybrid Search Ready"

)

Hybrid Search Ready


In [20]:
# ============================
# CELL 13: TEST RETRIEVAL
# ============================


results = hybrid_search(

    "How do I reset my password?",

    k=5

)



for i,r in enumerate(results):


    print("="*80)

    print(

        "RESULT",

        i+1

    )


    print()


    print(

        r["metadata"]["source"]

    )


    print()


    print(

        r["text"][:700]

    )

RESULT 1

HR-Management-Handbook-for-SAIs-2019.pdf

Commit to doing the right thing for 
the right reason, regardless of the circumstances  
M Resilient Adapt in the face of multiple changes while continuing to 
persevere toward the strategic goals of the SAI 
M Values driven Act from personal values and principles and motivate staff by 
connecting SAI goals to staff's personal values 
R Compassionate Care deeply about the well -being of others / the future / the 
environment and provide support to enhance the well-being of 
all  
R Serving Put the needs of clients, staff and communities first 
 
* Individual, Relationship, Motivational, Quality 
 
People Leadership  
 
Lead inspire and challenge others through decisive acti on, empowerment, 
RESULT 2

Pexip_Infinity_Release_Notes_v40.a.pdf

PexipInfinityv40ReleaseNotes Changesinfunctionality
©2026PexipAS Version40.a   April2026 Page10of14
Feature Description 
Deprecation of 
password-based 
authentication for 
the Teams 
Connector CVI

In [21]:
# ============================
# CELL 14: WATSONX CLIENT
# ============================


from ibm_watsonx_ai.foundation_models import ModelInference


llm = ModelInference(

    model_id="meta-llama/llama-3-3-70b-instruct",

    credentials={

        "apikey": "",

        "url": "https://eu-de.ml.cloud.ibm.com"

    },

    project_id=""

)


print(

"LLM Connected"

)

LLM Connected


In [22]:
# ============================
# CELL 15: ENTERPRISE AGENT
# ============================


def ask_company_ai(

        question

):


    docs = hybrid_search(

        question,

        k=5

    )



    context = ""


    for d in docs:


        context += f"""

SOURCE:

{d['metadata']['source']}


TEXT:

{d['text']}



------------------


"""



    prompt = f"""

You are Enterprise Knowledge Navigator.


Answer only from context.


If information cannot be found say


'Not found in company documents.'


CONTEXT


{context}



QUESTION


{question}



ANSWER



"""



    response = llm.chat(

        messages=[

            {

                "role":"user",

                "content":prompt

            }

        ]

    )


    return response["choices"][0]["message"]["content"]

In [23]:
# ==========================================
# CELL 16
# METADATA TAGGING AGENT
# ==========================================
#
# Automatically assigns metadata tags
# to every chunk.
#
# This improves filtering and retrieval.
#
# Examples:
#
# Department = HR
# Department = IT
# Department = Support
#
# Document Type = Policy
# Document Type = Manual
# Document Type = SOP
#
# Confidentiality = Public/Internal
#

def metadata_agent(chunk):

    text = chunk["text"].lower()

    department = "General"
    doc_type = "Document"
    confidentiality = "Internal"


    if "leave policy" in text:
        department = "HR"
        doc_type = "Policy"


    elif "password" in text:
        department = "IT"
        doc_type = "Security"


    elif "customer" in text:
        department = "Support"
        doc_type = "Guide"


    chunk["metadata"]["department"] = department
    chunk["metadata"]["doc_type"] = doc_type
    chunk["metadata"]["confidentiality"] = confidentiality


    return chunk


all_chunks = [

    metadata_agent(chunk)

    for chunk in all_chunks

]


print("Metadata tagging complete")

Metadata tagging complete


In [24]:
# ==========================================
# CELL 17
# FILTERED SEARCH
# ==========================================


def filtered_search(

        query,
        department=None,
        k=5
):


    results = hybrid_search(

        query,
        k=20
    )


    if department:


        results = [

            r

            for r in results

            if r["metadata"]["department"]==department

        ]


    return results[:k]

In [25]:
# ==========================================
# CELL 18
# DOCUMENT SUMMARIZER
# ==========================================


def summarize_document(text):


    prompt = f"""

Summarize this document.

Maximum 5 bullet points.


{text[:12000]}


"""


    response = llm.chat(

        messages=[

            {

                "role":"user",

                "content":prompt

            }

        ]

    )


    return response["choices"][0]["message"]["content"]




In [26]:
summary = summarize_document(

    documents[0]["text"]

)


print(summary)


Here are 5 key points summarizing the Unilever Annual Report and Accounts 2025:

* Unilever's turnover for 2025 was €50.5 billion, down 3.8% versus the previous year due to currency headwinds, but up 2.3% excluding currency impacts.
* The company delivered operating profit of €9.0 billion, or €10.1 billion on an underlying basis, and free cash flow of €5.9 billion, representing 100% cash conversion.
* Unilever's strategy focuses on three key areas: "Desire at Scale" to elevate brand offerings, "Play to Win" to build a high-performance culture, and "Fit for AI Age" to leverage technology and simplify the organization.
* The company has made progress towards its sustainability goals, with a focus on climate, nature, plastics, and livelihoods, and has sharpened its focus on seven strategic growth opportunities, including beauty, wellbeing, and digital commerce.
* Despite challenges in certain markets, such as Latin America, the company is moving in the right direction, with volume growth 

/opt/user-env/pyt6/lib64/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/opt/user-env/pyt6/lib64/python3.12/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The value of 'max_tokens' for this model was set to value 1024
ID: unspecified_max_token
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)


In [27]:
# ==========================================
# CELL 19
# CITATION AGENT
# ==========================================


def add_citations(docs):


    refs=[]


    for doc in docs:


        refs.append(

            f'{doc["metadata"]["source"]}'

        )


    return list(

        set(refs)

    )


In [28]:
docs = hybrid_search(

    "password reset"

)


print(

    add_citations(docs)

)


['Pexip_Infinity_Release_Notes_v40.a.pdf', '04-Policy-Manual-Apr-2016-excl-SAICA.pdf', 'Human_Resources_Manual.pdf', 'unilever-annual-report-and-accounts-2025.pdf']


In [29]:
# ==========================================
# CELL 20
# KNOWLEDGE GAP DETECTOR
# ==========================================


def knowledge_gap(docs):


    if len(docs)==0:

        return True


    return False


In [30]:
# ==========================================
# CELL 21
# SEARCH ANALYTICS
# ==========================================


search_history=[]


def log_query(query):


    search_history.append(

        query

    )


def analytics():



    from collections import Counter


    stats = Counter(

        search_history

    )


    return stats.most_common(10)




In [31]:

log_query(

"vpn password"

)

log_query(

"leave policy"

)

log_query(

"vpn password"

)



print(

analytics()

)


[('vpn password', 2), ('leave policy', 1)]


In [32]:
# ==========================================
# CELL 22
# MULTI AGENT ORCHESTRATOR
# ==========================================


def enterprise_agent(

        query
):


    log_query(query)


    docs = hybrid_search(


        query

    )


    if knowledge_gap(docs):


        return "No knowledge found."


    citations = add_citations(

        docs

    )


    answer = generate_answer(

        query

    )


    return {


        "answer":answer,


        "citations":citations


    }
